# Debug diagnostics and per-frame plots
Use the `debug` recorder when you need per-frame residual summaries and raw-stack statistics, then inspect the worst frame and the debug-specific plots inline. Before running the code, install the released package in this notebook's environment with `%pip install "fpm-rs[notebook]"`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import fpm_rs as fpm

In [ ]:
optics = fpm.Optics(532e-9, 0.10, 4.0, 6.5e-6)
geometry = fpm.PlanarLEDArray((3, 3), 4e-3, (1.0, 1.0), fpm.ArrayPose.from_translation((0.0, 0.0, -90e-3)))
illumination = fpm.Illumination(geometry)
model = fpm.compile_model(optics, illumination, (16, 16), (32, 32))

row, column = np.indices(model.reconstruction_shape)
amplitude = 0.65 + 0.35 * ((row // 4 + column // 4) % 2)
phase = 0.35 * np.sin(row / 5.0) * np.cos(column / 7.0)
object_field = np.asarray(amplitude * np.exp(1j * phase), dtype=np.complex128)

simulation = fpm.simulate(model, object_field, seed=23)
problem = fpm.ReconstructionProblem(
    simulation.measurements,
    simulation.reconstruction_model,
    name="diagnostics-debug",
)
recorder = fpm.DiagnosticRecorder("debug", every=2)
result = fpm.AlternatingProjection(iterations=6).run(
    problem,
    callbacks=[recorder],
)
diagnostics = recorder.diagnostics()
len(diagnostics["frame_diagnostics"]), len(diagnostics["raw_frame_statistics"])

In [ ]:
latest_iteration = max(entry["iteration"] for entry in diagnostics["frame_diagnostics"])
latest_frames = [
    entry
    for entry in diagnostics["frame_diagnostics"]
    if entry["iteration"] == latest_iteration
]
max(latest_frames, key=lambda entry: entry["normalized_l2"])

In [ ]:
plot_results = {
    "frame_residuals": fpm.plot.plot_frame_residuals(diagnostics),
    "raw_stack_stats": fpm.plot.plot_raw_stack_stats(diagnostics),
    "crop_indices": fpm.plot.plot_crop_indices(diagnostics),
    "residuals_on_fourier_centers": (
        fpm.plot.plot_residuals_on_fourier_centers(diagnostics)
    ),
}
for plot_name, plot in plot_results.items():
    if plot is None:
        continue
    figure, axes = plot
    display(figure)
    # figure.savefig(f"{plot_name}.png", dpi=180, bbox_inches="tight")
    plt.close(figure)

In [ ]:
diagnostics["iteration_diagnostics"][-1]